In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from config import RESULTS_P2, FIGURES_P2, COLORS

RED   = COLORS["red"]
BLUE  = COLORS["blue"]
GOLD  = COLORS["gold"]
GREY  = COLORS["grey"]
BG    = COLORS["bg"]
INK   = COLORS["ink"]


#  Fig P2-06: Odds Ratios Across Models
def fig_odds_ratios():
    path = RESULTS_P2 / "regression_model_results.csv"
    if not path.exists():
        print("  Skipping fig_p2_06 — run 02b_regression_chain.py first")
        return

    coefs = pd.read_csv(path)

    # Focus on pipeline and threat terms across Models B and D
    TERM_LABELS = {
        "threat_num":    "Threat Level (per unit)",
        "is_community":  "Community vs CAP",
        "is_287g":       "287(g) vs CAP",
        "is_other":      "Other vs CAP",
        "crim_num":      "Criminality (per unit)",
    }

    models = {
        "B_threat_plus_pipeline": "Model B\n(threat + pipeline)",
        "D_full_year_fe":         "Model D\n(full + year FE)",
    }

    fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor=BG)
    fig.suptitle(
        "Logistic Regression Odds Ratios: Pipeline Effect on Deportation\n"
        "CAP arrests = reference category | 95% confidence intervals shown",
        fontsize=12, fontweight="bold",
    )

    for i, (model_name, model_label) in enumerate(models.items()):
        ax = axes[i]
        ax.set_facecolor(BG)
        sub = coefs[
            (coefs["model"] == model_name) &
            (coefs["term"].isin(TERM_LABELS))
        ].copy()
        sub["label"] = sub["term"].map(TERM_LABELS)
        sub = sub.sort_values("odds_ratio")

        colors = []
        for _, row in sub.iterrows():
            if row["odds_ratio"] > 1:
                colors.append(RED)
            elif row["p_value"] < 0.05:
                colors.append(BLUE)
            else:
                colors.append(GREY)

        y = np.arange(len(sub))
        ax.barh(y, sub["odds_ratio"], color=colors, alpha=0.82, edgecolor="white")

        # CI bars
        for j, (_, row) in enumerate(sub.iterrows()):
            ax.plot(
                [row["or_ci_low"], row["or_ci_high"]], [j, j],
                color=INK, linewidth=1.5, alpha=0.6
            )

        ax.axvline(1.0, color=INK, linewidth=1.5, linestyle="--", alpha=0.7,
                   label="OR = 1 (no effect)")

        for j, (_, row) in enumerate(sub.iterrows()):
            stars = ("***" if row["p_value"] < 0.01 else
                     "**"  if row["p_value"] < 0.05 else
                     "*"   if row["p_value"] < 0.10 else "n.s.")
            ax.text(row["or_ci_high"] + 0.02, j,
                    f" OR={row['odds_ratio']:.3f} {stars}",
                    va="center", fontsize=8, color=INK)

        ax.set_yticks(y)
        ax.set_yticklabels(sub["label"], fontsize=10)
        ax.set_xlabel("Odds Ratio (reference: CAP pipeline)", fontsize=10)
        ax.set_title(model_label, fontsize=11, fontweight="bold")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="x", alpha=0.2, color=GREY)
        ax.set_xlim(0, sub["or_ci_high"].max() + 0.3)

    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig_p2_06_regression_odds_ratios.png",
                dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p2_06")


# Fig P2-07: Model Comparison + Mediation
def fig_model_comparison():
    path = RESULTS_P2 / "regression_model_results.csv"
    if not path.exists():
        print("  Skipping fig_p2_07 — run 02b_regression_chain.py first")
        return

    coefs = pd.read_csv(path)

    # Pseudo R² by model
    r2_data = (
        coefs.groupby("model")["pseudo_r2"]
        .first()
        .reset_index()
        .rename(columns={"model": "model_name"})
    )
    r2_data["label"] = r2_data["model_name"].map({
        "A_threat_only":         "Model A\nThreat only",
        "B_threat_plus_pipeline":"Model B\nThreat + Pipeline",
        "C_pipeline_only":       "Model C\nPipeline only",
        "D_full_year_fe":        "Model D\nFull + Year FE",
    })
    r2_data = r2_data.sort_values("pseudo_r2")

    # Community OR across models
    community_data = coefs[coefs["term"] == "is_community"].copy()
    community_data["label"] = community_data["model"].map({
        "B_threat_plus_pipeline": "Model B\n(+ threat)",
        "C_pipeline_only":        "Model C\n(pipeline only)",
        "D_full_year_fe":         "Model D\n(+ year FE + criminality)",
    })
    community_data = community_data.dropna(subset=["label"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), facecolor=BG)
    fig.suptitle(
        "Model Comparison: Variance Explained and Pipeline Effect Stability",
        fontsize=12, fontweight="bold",
    )

    # Left: Pseudo R²
    ax1 = axes[0]
    ax1.set_facecolor(BG)
    bar_colors = [BLUE if "pipeline" in r.lower() or "full" in r.lower() else GOLD
                  for r in r2_data["model_name"]]
    bars = ax1.barh(r2_data["label"], r2_data["pseudo_r2"] * 100,
                    color=bar_colors, alpha=0.85, edgecolor="white")
    for bar, (_, row) in zip(bars, r2_data.iterrows()):
        ax1.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                 f"{row['pseudo_r2']*100:.2f}%",
                 va="center", fontsize=9, color=INK)
    ax1.set_xlabel("McFadden Pseudo R² (%)", fontsize=10)
    ax1.set_title("Variance Explained by Model\nPipeline explains more than official threat score",
                  fontsize=10, fontweight="bold")
    ax1.spines[["top", "right"]].set_visible(False)
    ax1.grid(axis="x", alpha=0.2, color=GREY)

    # Right: Community OR stability
    ax2 = axes[1]
    ax2.set_facecolor(BG)
    y = np.arange(len(community_data))
    ax2.barh(y, community_data["odds_ratio"],
             color=BLUE, alpha=0.82, edgecolor="white")
    for j, (_, row) in enumerate(community_data.iterrows()):
        ax2.plot([row["or_ci_low"], row["or_ci_high"]], [j, j],
                 color=INK, linewidth=1.5, alpha=0.6)
        ax2.text(row["or_ci_high"] + 0.01, j,
                 f" OR={row['odds_ratio']:.3f}",
                 va="center", fontsize=9, color=INK)
    ax2.axvline(1.0, color=INK, linewidth=1.5, linestyle="--", alpha=0.7)
    ax2.set_yticks(y)
    ax2.set_yticklabels(community_data["label"], fontsize=10)
    ax2.set_xlabel("Odds Ratio: Community vs CAP", fontsize=10)
    ax2.set_title("Community Pipeline Effect Stability\nOR remains <0.5 across all specifications",
                  fontsize=10, fontweight="bold")
    ax2.spines[["top", "right"]].set_visible(False)
    ax2.grid(axis="x", alpha=0.2, color=GREY)

    # Mediation annotation
    c_coef = community_data.loc[community_data["model"] == "C_pipeline_only", "odds_ratio"].values
    b_coef = community_data.loc[community_data["model"] == "B_threat_plus_pipeline", "odds_ratio"].values
    if len(c_coef) and len(b_coef):
        attenuation = abs(np.log(c_coef[0]) - np.log(b_coef[0])) / abs(np.log(c_coef[0])) * 100
        ax2.text(0.05, len(community_data) - 0.3,
                 f"Threat mediates {attenuation:.0f}% of\npipeline effect on deportation",
                 fontsize=8.5, color=GREY, fontstyle="italic",
                 transform=ax2.get_yaxis_transform())

    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig_p2_07_model_comparison.png",
                dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p2_07")


# Fig P2-08: Reclassification Analysis
def fig_reclassification():
    path = RESULTS_P2 / "criminality_reclassification.csv"
    if not path.exists():
        print("  Skipping fig_p2_08 — run 02b_regression_chain.py first")
        return

    df = pd.read_csv(path)

    fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
    ax.set_facecolor(BG)

    y = np.arange(len(df))
    bars_m = ax.barh(y - 0.2, df["mismatch_rate"] * 100, 0.35,
                     color=RED, alpha=0.82, edgecolor="white",
                     label="Overall reclassification rate")
    bars_u = ax.barh(y + 0.2, df["upgrade_rate"] * 100, 0.35,
                     color=GOLD, alpha=0.82, edgecolor="white",
                     label="Upgraded (more severe at case level)")

    for bar, val in zip(bars_m, df["mismatch_rate"]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                f"{val:.1%}", va="center", fontsize=8.5, color=INK)
    for bar, val in zip(bars_u, df["upgrade_rate"]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                f"{val:.1%}", va="center", fontsize=8.5, color=INK)

    ax.set_yticks(y)
    ax.set_yticklabels(df["pipeline"], fontsize=10)
    ax.set_xlabel("Rate (%)", fontsize=11)
    ax.set_title(
        "Criminality Reclassification: Apprehension vs Case Label\n"
        "98.3% of records show identical classification at arrest and case resolution\n"
        "⚠ This near-perfect consistency raises data integrity questions",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=9, framealpha=0.9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.2, color=GREY)

    # Data integrity note
    ax.text(0.98, 0.04,
            "NOTE: Near-zero reclassification may indicate accurate initial\n"
            "classification, or records being carried forward without review.\n"
            "See limitations section for full discussion.",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=7.5, color=GREY, fontstyle="italic",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=BG, edgecolor=GREY, alpha=0.8))

    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig_p2_08_reclassification.png",
                dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p2_08")


def main():
    print("Generating Phase 2 regression figures...")
    fig_odds_ratios()
    fig_model_comparison()
    fig_reclassification()
    print(f"\nRegression figures saved to {FIGURES_P2}")


if __name__ == "__main__":
    main()